# 03 — Model Comparison & Winner Selection
**Benchmark Table | Efficiency Scores | Chemical Space**

This notebook:
1. Loads all saved model results from `./results/`
2. Builds the full benchmark table (accuracy + resource cost)
3. Computes efficiency score per model
4. Visualizes chemical space coverage
5. Selects the winner for optimization in `04_optimize.ipynb`

> **Prerequisite:** All 4 models in `02_baselines.ipynb` must have run.

## 0. Imports & Load Results

In [ ]:
import sys, json, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from utils.metrics import load_all_results, efficiency_score

with open('./config.json') as f:
    CONFIG = json.load(f)

results = load_all_results('./results/')

MODEL_LABELS = {
    'rf_morgan':    'RF + Morgan FP',
    'attentivefp':  'AttentiveFP',
    'mpnn':         'MPNN (DeepChem)',
    'chemberta2':   'ChemBERTa-2',
}

MODEL_COLORS = {
    'rf_morgan':    '#5C85D6',
    'attentivefp':  '#59B87A',
    'mpnn':         '#E8874A',
    'chemberta2':   '#C85BAF',
}

print(f'Loaded results for {len(results)} models:')
for k in results:
    print(f'  {MODEL_LABELS.get(k, k)}')

## 1. Full Benchmark Table

In [ ]:
rows = []
for model_id, data in results.items():
    rows.append({
        'Model':          MODEL_LABELS.get(model_id, model_id),
        'Test R²':        data.get('test_r2', np.nan),
        'Test RMSE':      data.get('test_rmse', np.nan),
        'Spearman ρ':     data.get('test_spearman_rho', np.nan),
        'Peak VRAM (MB)': data.get('peak_vram_mb', 0),
        'Train (min)':    data.get('train_time_min', np.nan),
        'CO2 (mg)':       round(data.get('kg_co2', 0) * 1e6, 3),
        'Efficiency ↑':   data.get('efficiency_score', np.nan),
        '_id':            model_id,
    })

df_bench = pd.DataFrame(rows).sort_values('Efficiency ↑', ascending=False)

print('Full Benchmark Table (sorted by Efficiency Score)')
print('=' * 80)
display_cols = [c for c in df_bench.columns if c != '_id']
print(df_bench[display_cols].to_string(index=False))
print()
print('Efficiency score = weighted combo of R², VRAM savings, time savings')
print('Higher = better value for compute invested.')

df_bench.to_csv('./results/benchmark_table.csv', index=False)

## 2. Benchmark Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Benchmark — RTX 3070 | EGFR Kinase', fontsize=14, fontweight='bold')

model_names = [MODEL_LABELS.get(r['_id'], r['_id']) for _, r in df_bench.iterrows()]
colors      = [MODEL_COLORS.get(r['_id'], '#888') for _, r in df_bench.iterrows()]

# ── Test R² ─────────────────────────────────────────────────────
ax = axes[0, 0]
r2_vals = df_bench['Test R²'].values
bars = ax.bar(model_names, r2_vals, color=colors, edgecolor='white', linewidth=0.8)
ax.set_ylabel('R²')
ax.set_title('Test R² (higher = better)')
ax.set_ylim(0, 1)
for bar, val in zip(bars, r2_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)
ax.tick_params(axis='x', rotation=15)

# ── Peak VRAM ───────────────────────────────────────────────────
ax = axes[0, 1]
vram_vals = df_bench['Peak VRAM (MB)'].values
bars = ax.bar(model_names, vram_vals, color=colors, edgecolor='white', linewidth=0.8)
ax.axhline(y=8000, color='red', linestyle='--', alpha=0.7, label='RTX 3070 limit (8GB)')
ax.set_ylabel('Peak VRAM (MB)')
ax.set_title('Peak GPU Memory (lower = better)')
ax.legend(fontsize=8)
for bar, val in zip(bars, vram_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9)
ax.tick_params(axis='x', rotation=15)

# ── Training Time ────────────────────────────────────────────────
ax = axes[1, 0]
time_vals = df_bench['Train (min)'].values
bars = ax.bar(model_names, time_vals, color=colors, edgecolor='white', linewidth=0.8)
ax.set_ylabel('Minutes')
ax.set_title('Training Time (lower = better)')
for bar, val in zip(bars, time_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.1f}m', ha='center', va='bottom', fontsize=9)
ax.tick_params(axis='x', rotation=15)

# ── Efficiency Score (composite) ─────────────────────────────────
ax = axes[1, 1]
eff_vals = df_bench['Efficiency ↑'].values
bars = ax.bar(model_names, eff_vals, color=colors, edgecolor='white', linewidth=0.8)
ax.set_ylabel('Efficiency Score')
ax.set_title('Efficiency Score — accuracy ÷ resource cost (higher = better)')
ax.set_ylim(0, 1)
for bar, val in zip(bars, eff_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)
ax.tick_params(axis='x', rotation=15)

# Highlight winner
winner_idx = np.argmax(eff_vals)
bars[winner_idx].set_edgecolor('gold')
bars[winner_idx].set_linewidth(2.5)
ax.text(0.5, 0.95, '★ Winner', transform=ax.transAxes,
        ha='center', va='top', color='goldenrod', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('./results/03_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to results/03_benchmark.png')

## 3. Accuracy vs Cost Scatter
The sweet spot is top-right (high R², low VRAM).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for _, row in df_bench.iterrows():
    mid   = row['_id']
    label = MODEL_LABELS.get(mid, mid)
    color = MODEL_COLORS.get(mid, '#888')
    vram  = max(row['Peak VRAM (MB)'], 10)  # avoid zero for CPU model
    r2    = row['Test R²']
    time  = row['Train (min)']

    sc = ax.scatter(vram, r2, s=time*30, color=color, alpha=0.85,
                    edgecolors='white', linewidth=1.2, zorder=3)
    ax.annotate(label, (vram, r2), textcoords='offset points',
                xytext=(8, 4), fontsize=9)

ax.set_xlabel('Peak VRAM (MB) — lower is better →', fontsize=11)
ax.set_ylabel('Test R² — higher is better ↑', fontsize=11)
ax.set_title('Accuracy vs GPU Memory\n(bubble size = training time)', fontsize=12)
ax.axhline(y=0.7, color='gray', linestyle=':', alpha=0.5, label='R²=0.7 threshold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Annotate ideal region
ax.text(0.02, 0.95, '← Ideal region\n(high accuracy, low VRAM)',
        transform=ax.transAxes, fontsize=9, color='green', va='top')

plt.tight_layout()
plt.savefig('./results/03_accuracy_vs_cost.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Chemical Space — UMAP Visualization

In [ ]:
import umap
from utils.mol_utils import batch_smiles_to_morgan
import pandas as pd

df_all = pd.read_csv('./data/full_clean.csv')

# Sample 3000 for speed
df_sample = df_all.sample(n=min(3000, len(df_all)), random_state=42)

print('Computing Morgan fingerprints for UMAP...')
X_fps, valid_idx = batch_smiles_to_morgan(df_sample['Drug'].tolist(), radius=2, n_bits=1024)
y_vals = df_sample['Y_log'].iloc[valid_idx].values

print(f'Running UMAP on {len(X_fps)} compounds (2D projection)...')
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                    metric='jaccard', random_state=42)
embedding = reducer.fit_transform(X_fps)

# Plot
fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(embedding[:, 0], embedding[:, 1],
               c=y_vals, cmap='RdYlGn_r', alpha=0.5, s=6)
plt.colorbar(sc, ax=ax, label='log(IC50) — red=active, green=inactive')

# Overlay known EGFR drugs
with open('./data/known_drugs.json') as f:
    known_drugs = json.load(f)

known_fps, _ = batch_smiles_to_morgan(list(known_drugs.values()), radius=2, n_bits=1024)
if len(known_fps) > 0:
    known_emb = reducer.transform(known_fps)
    ax.scatter(known_emb[:, 0], known_emb[:, 1],
               color='blue', s=120, marker='*', zorder=5, label='Known EGFR drugs')
    for i, name in enumerate(known_drugs.keys()):
        if i < len(known_emb):
            ax.annotate(name, known_emb[i], fontsize=8, color='blue',
                        textcoords='offset points', xytext=(5, 5))

ax.set_title('EGFR Chemical Space — UMAP (Morgan FP, Jaccard distance)', fontsize=12)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('./results/03_umap_chemical_space.png', dpi=150, bbox_inches='tight')
plt.show()
print('UMAP saved to results/03_umap_chemical_space.png')

## 5. Winner Selection

In [ ]:
winner_row = df_bench.iloc[0]
winner_id  = winner_row['_id']
winner_name = MODEL_LABELS.get(winner_id, winner_id)

print('=' * 55)
print('WINNER SELECTION')
print('=' * 55)
print(f'  Model:         {winner_name}')
print(f'  Test R²:       {winner_row["Test R²"]:.4f}')
print(f'  Test RMSE:     {winner_row["Test RMSE"]:.4f}')
print(f'  Spearman ρ:    {winner_row["Spearman ρ"]:.4f}')
print(f'  Peak VRAM:     {winner_row["Peak VRAM (MB)"]:.0f} MB')
print(f'  Train time:    {winner_row["Train (min)"]:.1f} min')
print(f'  CO2:           {winner_row["CO2 (mg)"]:.3f} mg')
print(f'  Efficiency ↑:  {winner_row["Efficiency ↑"]:.4f}')
print()
print('This model will be optimized in 04_optimize.ipynb')
print('Optimizations to apply:')
print('  1. Mixed precision (AMP)  — already enabled during training')
print('  2. Gradient checkpointing — memory vs compute trade-off')
print('  3. Knowledge distillation — train smaller student model')
print('  4. INT8 quantization      — 4x memory at inference time')
print('  5. Optuna HPO             — find smallest model hitting R² target')

# Save winner to config
CONFIG['winner_model'] = winner_id
CONFIG['winner_name']  = winner_name
with open('./config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2, default=str)

print(f'\nWinner saved to config.json → winner_model: {winner_id}')
print('\n✓ Ready for 04_optimize.ipynb')